# 00 · Raw data tour and contract comparison

Blocks 3 and 4 of week 1. Exploration only: nothing here feeds the pipeline.

Run `uv run lobmm convert --verify` first; this notebook reads the Parquet files.

**Rule:** only the dev day (a research day) is explored here. Never open a locked test day.

1. Row counts and file sizes
2. The first snapshot and rows before it
3. Deletions (`amount = 0`)
4. Feed delay (`local_timestamp - timestamp`)
5. Updates that share a timestamp
6. Tick size, tick ÷ price, spread in ticks, depth at the best prices
7. Summary and decision

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

from lobmm.config import REPO_ROOT, load_config

cfg = load_config()
DAY = cfg["days"]["dev_day"]  # a research day; test days stay locked
assert DAY not in cfg["days"]["test"]
SYMBOLS = cfg["market"]["candidates"]
PQ = REPO_ROOT / cfg["data"]["parquet_dir"]
RAW = REPO_ROOT / cfg["data"]["raw_dir"]

con = duckdb.connect()
con.execute("SET memory_limit = '6GB'")  # spills to disk instead of running out of RAM
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


def pq(data_type: str, symbol: str, day: str = DAY) -> str:
    p = PQ / data_type / symbol / f"{day}.parquet"
    assert p.exists(), f"missing {p}: run `uv run lobmm convert` first"
    return p.as_posix()


def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()


print(DAY, SYMBOLS)

## 1. Row counts and file sizes

In [ ]:
rows = []
for sym in SYMBOLS:
    for data_type in cfg["data"]["data_types"]:
        p = pq(data_type, sym)
        n = con.sql(f"SELECT count(*) FROM '{p}'").fetchone()[0]
        raw = next(RAW.rglob(f"*_{data_type}_{DAY}_{sym}.csv.gz"))
        rows.append(
            {
                "symbol": sym,
                "data_type": data_type,
                "rows": n,
                "csv_gz_mb": raw.stat().st_size / 2**20,
                "parquet_mb": Path(p).stat().st_size / 2**20,
            }
        )
counts = pd.DataFrame(rows)
counts["parquet_vs_raw"] = counts.parquet_mb / counts.csv_gz_mb
counts

## 2. The first snapshot

`seq` is the row's position in the original file, so the first snapshot row's `seq` is also
the number of rows before it (these get skipped in the rebuild).

`snapshot_starts` counts every place a snapshot begins after normal updates. More than 1 means
the book was reset mid-day (a reconnect), and the rebuild must clear and restart there.

In [ ]:
snap = []
for sym in SYMBOLS:
    p = pq("incremental_book_L2", sym)
    snap.append(
        q(f"""
        WITH b AS (
            SELECT seq, is_snapshot,
                   lag(is_snapshot) OVER (ORDER BY seq) AS prev_snapshot
            FROM '{p}'
        )
        SELECT '{sym}' AS symbol,
               min(seq) FILTER (WHERE is_snapshot) AS first_snapshot_seq,
               count(*) FILTER (WHERE is_snapshot AND coalesce(NOT prev_snapshot, true))
                   AS snapshot_starts,
               count(*) FILTER (WHERE is_snapshot) AS snapshot_rows
        FROM b
        """)
    )
snap = pd.concat(snap, ignore_index=True)
snap["rows_before_first_snapshot"] = snap.first_snapshot_seq
snap

In [ ]:
# When does each snapshot block start? (UTC)
for sym in SYMBOLS:
    p = pq("incremental_book_L2", sym)
    starts = q(f"""
        WITH b AS (
            SELECT seq, timestamp, is_snapshot,
                   lag(is_snapshot) OVER (ORDER BY seq) AS prev_snapshot
            FROM '{p}'
        )
        SELECT seq, to_timestamp(timestamp / 1e6) AS exchange_time_utc
        FROM b
        WHERE is_snapshot AND coalesce(NOT prev_snapshot, true)
        ORDER BY seq
        """)
    print(sym)
    print(starts.to_string(index=False), end="\n\n")

## 3. Deletions (`amount = 0`)

`amount` is the level's new total size, not a change. Zero means: remove the level.

In [ ]:
dels = []
for sym in SYMBOLS:
    p = pq("incremental_book_L2", sym)
    dels.append(
        q(f"""
        SELECT '{sym}' AS symbol, side, is_snapshot,
               count(*) AS updates,
               count(*) FILTER (WHERE amount = 0) AS deletions,
               avg((amount = 0)::INT) AS deletion_share
        FROM '{p}'
        GROUP BY side, is_snapshot
        ORDER BY side, is_snapshot
        """)
    )
pd.concat(dels, ignore_index=True)

In [ ]:
# A few deletions in context: the update before, the deletion, the update after
sym = SYMBOLS[0]
p = pq("incremental_book_L2", sym)
q(f"""
    WITH d AS (SELECT seq FROM '{p}' WHERE amount = 0 AND NOT is_snapshot LIMIT 3)
    SELECT b.seq, b.local_timestamp, b.side, b.price, b.amount
    FROM '{p}' b JOIN d ON b.seq BETWEEN d.seq - 1 AND d.seq + 1
    ORDER BY b.seq
""")

## 4. Feed delay

`local_timestamp - timestamp`, in milliseconds: how long a message took to arrive. It is a free
lower bound on realistic latency for the backtester. Negative values mean the two clocks
disagree slightly.

In [ ]:
delay = []
for sym in SYMBOLS:
    for data_type in ("incremental_book_L2", "trades"):
        p = pq(data_type, sym)
        delay.append(
            q(f"""
            SELECT '{sym}' AS symbol, '{data_type}' AS data_type,
                   quantile_cont(d, 0.50) AS p50_ms,
                   quantile_cont(d, 0.90) AS p90_ms,
                   quantile_cont(d, 0.99) AS p99_ms,
                   max(d) AS max_ms,
                   avg((d < 0)::INT) AS share_negative
            FROM (SELECT (local_timestamp - timestamp) / 1000.0 AS d FROM '{p}')
            """)
        )
delay = pd.concat(delay, ignore_index=True)
delay

## 5. Updates that share a timestamp

The exchange sends several level updates in one message, so many rows share a
`local_timestamp`. The rebuild must apply a whole group before checking the book
(best bid < best ask), because the book can look crossed halfway through a group.

Also checks that `local_timestamp` never goes backwards in file order.

In [ ]:
groups = []
for sym in SYMBOLS:
    p = pq("incremental_book_L2", sym)
    groups.append(
        q(f"""
        WITH g AS (
            SELECT local_timestamp, count(*) AS n
            FROM '{p}' WHERE NOT is_snapshot GROUP BY local_timestamp
        )
        SELECT '{sym}' AS symbol,
               count(*) AS messages,
               avg(n) AS mean_rows_per_msg,
               median(n) AS median_rows,
               quantile_cont(n, 0.99) AS p99_rows,
               max(n) AS max_rows,
               avg((n > 1)::INT) AS share_multi_row
        FROM g
        """)
    )
pd.concat(groups, ignore_index=True)

In [ ]:
order = []
for sym in SYMBOLS:
    p = pq("incremental_book_L2", sym)
    order.append(
        q(f"""
        WITH b AS (
            SELECT local_timestamp, timestamp,
                   lag(local_timestamp) OVER (ORDER BY seq) AS prev_local,
                   lag(timestamp) OVER (ORDER BY seq) AS prev_exchange
            FROM '{p}'
        )
        SELECT '{sym}' AS symbol,
               count(*) FILTER (WHERE local_timestamp < prev_local) AS local_ts_goes_back,
               count(*) FILTER (WHERE timestamp < prev_exchange) AS exchange_ts_goes_back
        FROM b
        """)
    )
pd.concat(order, ignore_index=True)

## 6. Tick size, tick ÷ price, spread and depth

- **Tick**: the smallest gap between distinct prices seen in the book.
- **Tick ÷ price**: prefer contracts at **0.3 bp or more**, where imbalance is most informative.
- **Spread in ticks** from the top-25 snapshots: ideally the spread is 1 tick most of the time.
- **Depth at the best prices**: median size at the best bid and ask, in coins and in USD.

In [ ]:
facts = []
for sym in SYMBOLS:
    book, trades, snap25 = (
        pq("incremental_book_L2", sym),
        pq("trades", sym),
        pq("book_snapshot_25", sym),
    )
    tick_row = q(f"""
        WITH p AS (SELECT DISTINCT round(price, 8) AS price FROM '{book}'),
             d AS (SELECT round(price - lag(price) OVER (ORDER BY price), 8) AS gap FROM p)
        SELECT min(gap) FILTER (WHERE gap > 0) AS tick,
               mode(gap) FILTER (WHERE gap > 0) AS most_common_gap
        FROM d
    """).iloc[0]
    tick = float(tick_row.tick)
    px = q(f"SELECT avg(price) AS avg_px, count(*) AS trades FROM '{trades}'").iloc[0]
    spread = q(f"""
        SELECT round((ask_px_0 - bid_px_0) / {tick}) AS spread_ticks,
               bid_sz_0, ask_sz_0, (ask_px_0 + bid_px_0) / 2 AS mid
        FROM '{snap25}'
        WHERE ask_px_0 IS NOT NULL AND bid_px_0 IS NOT NULL
    """)
    facts.append(
        {
            "symbol": sym,
            "tick": tick,
            "most_common_gap": float(tick_row.most_common_gap),
            "avg_trade_px": float(px.avg_px),
            "tick_bp": tick / float(px.avg_px) * 1e4,
            "trades_per_day": int(px.trades),
            "share_spread_1_tick": (spread.spread_ticks == 1).mean(),
            "median_spread_ticks": spread.spread_ticks.median(),
            "p90_spread_ticks": spread.spread_ticks.quantile(0.9),
            "share_crossed_or_locked": (spread.spread_ticks <= 0).mean(),
            "median_l1_depth_coins": pd.concat([spread.bid_sz_0, spread.ask_sz_0]).median(),
            "median_l1_depth_usd": pd.concat(
                [spread.bid_sz_0 * spread.mid, spread.ask_sz_0 * spread.mid]
            ).median(),
        }
    )
facts = pd.DataFrame(facts).set_index("symbol")
facts.T

In [ ]:
# Spread distribution in ticks, per contract (share of snapshots)
for sym in SYMBOLS:
    snap25 = pq("book_snapshot_25", sym)
    tick = facts.loc[sym, "tick"]
    dist = q(f"""
        SELECT round((ask_px_0 - bid_px_0) / {tick}) AS spread_ticks, count(*) AS n
        FROM '{snap25}' WHERE ask_px_0 IS NOT NULL AND bid_px_0 IS NOT NULL
        GROUP BY 1 ORDER BY 1
    """)
    dist["share"] = dist.n / dist.n.sum()
    print(sym)
    print(dist.head(6).to_string(index=False), end="\n\n")

## 7. Summary and decision

In [ ]:
MIN_BP = cfg["market"]["min_tick_over_price_bp"]
cols = ["tick", "avg_trade_px", "tick_bp", "share_spread_1_tick"]
summary = facts[cols + ["median_l1_depth_usd", "trades_per_day"]].copy()
summary["book_updates"] = counts[counts.data_type == "incremental_book_L2"].set_index("symbol")[
    "rows"
]
summary["feed_delay_p50_ms"] = delay[delay.data_type == "incremental_book_L2"].set_index("symbol")[
    "p50_ms"
]
summary["passes_tick_rule"] = summary.tick_bp >= MIN_BP
summary

In [ ]:
ok = summary[summary.passes_tick_rule].sort_values(
    ["share_spread_1_tick", "tick_bp"], ascending=False
)
print(f"Rule: tick / price >= {MIN_BP} bp, and the spread is 1 tick most of the time.\n")
for sym, row in summary.iterrows():
    verdict = "PASS" if row.passes_tick_rule else "fail"
    print(
        f"{sym:8s} tick={row.tick:g}  tick/price={row.tick_bp:.2f} bp  "
        f"1-tick spread {row.share_spread_1_tick:.0%}  -> {verdict}"
    )
print("\nCandidate main contract:", ok.index[0] if len(ok) else "none pass; discuss")

## Notes (fill in)

- Rows before the first snapshot:
- Mid-day snapshot resets:
- Deletion share:
- Feed delay p50 / p99:
- Typical rows per message:
- Tick ÷ price per contract:
- Main contract choice and why: